In [1]:
from maskllm.maskllm import MaskedLinear, MaskedLinearFrozen
from maskllm.sparsegpt import SparseGPT
import timm

/work/classtmp/dhawal04/mac_pruning_fv/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model_name = 'deit_tiny_patch16_224.fb_in1k'
model = timm.create_model('deit_tiny_patch16_224.fb_in1k', pretrained=True, num_classes=10)

In [3]:
def count_parameters(model):
    """
    Counts the number of trainable parameters in a PyTorch model.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [4]:
count_parameters(model)

5526346

In [5]:
import torch
import torch_pruning as tp

def print_model_mac(model):
    example_inputs = (torch.randn(1, 3, 224, 224).to(next(model.parameters()).device),)
    total_macs, _ = tp.utils.count_ops_and_params(model, example_inputs)
    print(f"[📊] Total model MACs: {total_macs/1e9:.3f}G")

In [6]:
print_model_mac(model)

[📊] Total model MACs: 1.259G


In [ ]:
from utils.analysis_isomorphism import ViTIsomorphicAnalyzer

isomorphic_analyser = ViTIsomorphicAnalyzer(model)
groups = isomorphic_analyser.create_isomorphic_groups(
    target_macs=0.6*1e9,
    baseline_macs=1.25*1e9
    )

[🔍] Found 12 transformer blocks
[📊] Total model MACs: 1.259G
[📊] Target MACs: 0.600G
[📊] Target ratio: 0.520
[🔍] Found 24 MLP layers (3,538,944 total params)
[📊] MLP MAC fraction estimated: 0.666 (3,538,944/5,310,336 params)
[🔍] Found 24 attention layers (1,769,472 total params)
[📊] Attention MAC fraction estimated: 0.333 (1,769,472/5,310,336 params)
[🪫] Max feasible reduction under caps: 0.990 → best achievable ≈ 0.013G
[🎯] LLM suggests: mlp_pruning=0.400, attn_pruning=0.150
[🎯] Using as pruning ratios: r_mlp=0.400, r_attn=0.150
[🎯] Predicted achievement: 0.317 (target: 0.520)
[⚖️] Post-alloc solve -> r_mlp=0.400, r_attn=0.150 (mlp_frac=0.667, attn_frac=0.333, caps: 0.99/0.99)
[📊] MLP: 0.839G → 0.503G (ratio: 0.400)
[📊] Attention: 0.420G → 0.357G (ratio: 0.150)
[📊] Final group summary:
  - MLP blocks: 12 couples
  - Attention blocks: 12 couples
  - Output projections: 0 layers


In [8]:
from pprint import pprint

pprint(groups)

{'attention_blocks': IsomorphicGroup(name='Attention Blocks (Coupled)',
                                     layers=[AttentionCouple(qkv=Linear(in_features=192, out_features=576, bias=True),
                                                             proj=Linear(in_features=192, out_features=192, bias=True),
                                                             qkv_name='blocks.0.attn.qkv',
                                                             proj_name='blocks.0.attn.proj'),
                                             AttentionCouple(qkv=Linear(in_features=192, out_features=576, bias=True),
                                                             proj=Linear(in_features=192, out_features=192, bias=True),
                                                             qkv_name='blocks.1.attn.qkv',
                                                             proj_name='blocks.1.attn.proj'),
                                             AttentionCouple(qkv=Linear(in_featu

In [9]:
from data.loaders import get_cifar10_loaders_pbench

batch_size = 1

train_loader, val_loader = get_cifar10_loaders_pbench(batch_size, num_workers=16)

/work/classtmp/dhawal04/mac_pruning_fv/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [10]:
import copy

original_model = copy.deepcopy(model)

In [11]:
pprint(original_model)

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=192, out_features=576, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=192, out_features=192, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=192, out_features=768, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)


In [ ]:
import torch
import torch.nn as nn

def replace_linear_with_(model, new_class, exclude=[], groups=None, **kwargs):
    """Replace linear layers with new_class in a model. It's an inplace operation.
    
    Args:
        model: The model to modify
        new_class: The class to replace Linear layers with (MaskedLinear)
        exclude_names: List of module names to exclude from replacement
        groups: Dictionary of IsomorphicGroup objects with pruning ratios
        **kwargs: Additional arguments for new_class
    """
    # Default N:M pattern if no groups specified
    default_N = kwargs.get('N', 2)
    default_M = kwargs.get('M', 4)
    
    def get_sparsity_config(full_name):
        """Determine N:M pattern based on full layer name"""
        if groups is None:
            return {'N': default_N, 'M': default_M}
        
        # Extract layer type from full name
        # Example: 'blocks.0.attn.qkv' or 'blocks.0.mlp.fc1'
        name_parts = full_name.split('.')
        
        # Map to group based on structure
        if len(name_parts) >= 4:
            if 'attn' in name_parts:
                if 'qkv' in name_parts or 'proj' in name_parts:
                    # Attention layers (qkv or proj)
                    if hasattr(groups, 'attention_blocks'):
                        ratio = groups.attention_blocks.pruning_ratio
                    elif 'attention_blocks' in groups:
                        ratio = groups['attention_blocks'].pruning_ratio
                    else:
                        ratio = 0.15  # Default for attention
                    return ratio_to_nm(ratio)
            elif 'mlp' in name_parts:
                if 'fc1' in name_parts or 'fc2' in name_parts:
                    # MLP layers
                    if hasattr(groups, 'mlp_blocks'):
                        ratio = groups.mlp_blocks.pruning_ratio
                    elif 'mlp_blocks' in groups:
                        ratio = groups['mlp_blocks'].pruning_ratio
                    else:
                        ratio = 0.4  # Default for MLP
                    return ratio_to_nm(ratio)
        
        # Output projections (head, etc.)
        if 'head' in full_name or 'fc' == name_parts[-1]:
            if hasattr(groups, 'output_projections'):
                ratio = groups.output_projections.pruning_ratio
            elif 'output_projections' in groups:
                ratio = groups['output_projections'].pruning_ratio
            else:
                ratio = 0.0  # Default no pruning
            return ratio_to_nm(ratio)
        
        return {'N': default_N, 'M': default_M}
    
    def ratio_to_nm(pruning_ratio):
        """Convert pruning ratio to N:M pattern"""
        # Density = 1 - pruning_ratio
        density = 1 - pruning_ratio

        # Map density to N:M patterns 
        # TODO: Need to explore more options of M values
        if density >= 0.8:   # ≤20% pruning
            return {'N': 4, 'M': 5}  # 80% dense
        elif density >= 0.75:  # ~25% pruning
            return {'N': 3, 'M': 4}  # 75% dense
        elif density >= 0.5:   # ~50% pruning
            return {'N': 2, 'M': 4}  # 50% dense
        elif density >= 0.4:   # ~60% pruning
            return {'N': 2, 'M': 5}  # 40% dense
        else:                  # High pruning
            return {'N': 1, 'M': 4}  # 25% dense
    
    def recursive_replace(module, prefix=''):
        """Recursively replace linear layers"""
        for name, child in module.named_children():
            # Build full name with prefix
            full_name = f"{prefix}.{name}" if prefix else name
            
            # Skip if in exclude list
            if full_name in exclude or name in exclude:
                continue
            
            if isinstance(child, nn.Linear):
                # Get sparsity config for this layer
                config = get_sparsity_config(full_name)
                
                # Prepare kwargs for new layer
                layer_kwargs = kwargs.copy()
                layer_kwargs.update(config)
                
                # Create new masked linear layer
                new_layer = new_class(
                    in_features=child.in_features,
                    out_features=child.out_features,
                    bias=child.bias is not None,
                    **layer_kwargs
                )
                
                # Copy weights and bias
                new_layer.weight.data = child.weight.data.clone()
                if child.bias is not None:
                    new_layer.bias.data = child.bias.data.clone()
                
                # Move to same device
                new_layer.to(child.weight.device)
                
                # Replace in parent module
                setattr(module, name, new_layer)
                
                print(f"Replaced {full_name}: {config}")
                
            else:
                # Recursively process children
                recursive_replace(child, full_name)
    
    # Start recursive replacement
    recursive_replace(model)
    return model

In [13]:
# model = copy.deepcopy(original_model)

In [14]:
# from maskllm.utils import replace_linear_with_
replace_linear_with_(model, MaskedLinearFrozen, exclude=[model.head], groups=groups)

Replaced blocks.0.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.0.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.0.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.0.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.1.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.1.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.1.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.1.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.2.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.2.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.2.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.2.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.3.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.3.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.3.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.3.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.4.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.4.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.4.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.4.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.5.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.5.attn.proj: {'N': 4, 'M': 5}
Replaced block

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): MaskedLinearFrozen(192, 576, bias=True, N=4, M=5)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): MaskedLinearFrozen(192, 192, bias=True, N=4, M=5)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): MaskedLinearFrozen(192, 768, bias=True, N=2, M=4)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (

In [15]:
count_parameters(model)

5526346

In [16]:
print_model_mac(model)

[📊] Total model MACs: 0.562G


In [ ]:
# sparsegpt actual

import torch
import transformers
import torch.nn as nn 

def find_layers(module, layers=[nn.Linear], name=''):
    """
    Recursively find the layers of a certain type in a module.

    Args:
        module (nn.Module): PyTorch module.
        layers (list): List of layer types to find.
        name (str): Name of the module.

    Returns:
        dict: Dictionary of layers of the given type(s) within the module.
    """
    if isinstance(module, tuple(layers)):
        return {name: module}
    res = {}
    for name1, child in module.named_children():
        res.update(find_layers(
            child, layers=layers, name=name + '.' + name1 if name != '' else name1
        ))
    return res


@torch.no_grad()  
def prune_sparsegpt(model, loader, nsamples=128, batch_size=1, device=torch.device("cuda:0"),   
                   prune_n=0, prune_m=0, sparsity_ratio=0.0, disable_update=False, groups=None):  
    # Initialize input caches (unchanged)  
    model.to(device)
    layers = model.blocks  
    dtype = next(iter(model.parameters())).dtype  
    inps = torch.zeros(  
        (nsamples, model.num_prefix_tokens+model.patch_embed.num_patches, model.embed_dim), dtype=dtype, device=device  
    )  
    cache = {'i': 0}  
  
    class Catcher(nn.Module):  
        def __init__(self, module):  
            super().__init__()  
            self.module = module  
        def forward(self, inp, **kwargs):  
            inps[cache['i']] = inp  
            cache['i'] += 1
            raise ValueError  
    layers[0] = Catcher(layers[0])  
    for batch in loader:  
        if cache['i'] == nsamples: break  
        try:  
            model(batch[0].to(device))  
        except ValueError:  
            pass  
    layers[0] = layers[0].module  
    torch.cuda.empty_cache()  
  
    outs = torch.zeros_like(inps)  
    print('Ready.')  
      
    for i in range(len(layers)):  
        layer = layers[i]  
        inps, outs = inps.to(device), outs.to(device)  
  
        subset = find_layers(layer)  
  
        gpts = {}  
        for name in subset:  
            gpts[name] = SparseGPT(subset[name])  
  
        def add_batch(name):  
            def tmp(_, inp, out):  
                gpts[name].add_batch(inp[0].data, out.data)  
            return tmp  
  
        handles = []  
        for name in gpts:  
            handles.append(subset[name].register_forward_hook(add_batch(name)))  
  
        for j in range(nsamples):  
            outs[j] = layer(inps[j].unsqueeze(0))[0]  
        for h in handles:  
            h.remove()  
  
        for name in gpts:  
            print(i, name)  
            print('Pruning ...')  
              
            # Get layer-specific N:M pattern  
            layer_n, layer_m = get_layer_sparsity(name, groups, prune_n, prune_m)  
              
            gpts[name].fasterprune(sparsity_ratio, prunen=layer_n, prunem=layer_m,   
                                  percdamp=0.01, blocksize=128, disable_update=disable_update)  
            gpts[name].free()  
  
        for j in range(nsamples):
            outs[j] = layer(inps[j].unsqueeze(0))[0]  
  
        layers[i] = layer   
        torch.cuda.empty_cache()  
  
        inps, outs = outs, inps  
  
    torch.cuda.empty_cache()
  
  
def get_layer_sparsity(layer_name, groups, default_n, default_m):  
    """Get N:M pattern for a specific layer based on groups"""  
    if groups is None:  
        return default_n, default_m  
      
    # Extract layer type from name  
    name_parts = layer_name.split('.')  
      
    # Check if this is an attention layer  
    if 'attn' in name_parts:  
        if 'qkv' in name_parts or 'proj' in name_parts:  
            if 'attention_blocks' in groups:  
                ratio = groups['attention_blocks'].pruning_ratio  
                return ratio_to_nm(ratio)  
      
    # Check if this is an MLP layer  
    elif 'mlp' in name_parts:  
        if 'fc1' in name_parts or 'fc2' in name_parts:  
            if 'mlp_blocks' in groups:  
                ratio = groups['mlp_blocks'].pruning_ratio  
                return ratio_to_nm(ratio)  
      
    # Default  
    return default_n, default_m  
  
  
def ratio_to_nm(pruning_ratio):  
    """Convert pruning ratio to N:M pattern"""  
    density = 1 - pruning_ratio  
      
    if density >= 0.8:   # ≤20% pruning  
        return 4, 5  # 80% dense  
    elif density >= 0.75:  # ~25% pruning  
        return 3, 4  # 75% dense  
    elif density >= 0.5:   # ~50% pruning  
        return 2, 4  # 50% dense  
    elif density >= 0.4:   # ~60% pruning  
        return 2, 5  # 40% dense  
    else:                  # High pruning  
        return 1, 4  # 25% dense
    

In [18]:
# import torch
# import transformers
# import torch.nn as nn 

# def find_layers(module, layers=[nn.Linear], name=''):
#     """
#     Recursively find the layers of a certain type in a module.

#     Args:
#         module (nn.Module): PyTorch module.
#         layers (list): List of layer types to find.
#         name (str): Name of the module.

#     Returns:
#         dict: Dictionary of layers of the given type(s) within the module.
#     """
#     if isinstance(module, tuple(layers)):
#         return {name: module}
#     res = {}
#     for name1, child in module.named_children():
#         res.update(find_layers(
#             child, layers=layers, name=name + '.' + name1 if name != '' else name1
#         ))
#     return res

# # def prune_sparsegpt(model, loader, nsamples=128, batch_size=1, device=torch.device("cuda:0"), groups=None, sparsity_ratio=0.0, disable_update=False):
# #     # If groups is provided, we'll use dynamic N:M based on layer groups
# #     # Otherwise fall back to original behavior
# #     model.to(device)
# #     # Helper function to convert pruning ratio to N:M pattern
# #     def ratio_to_nm(pruning_ratio):
# #         """Convert pruning ratio to N:M pattern"""
# #         # Common N:M patterns and their densities
# #         patterns = {
# #             0.02: (5, 6),   # 83% dense
# #             0.08: (4, 5),   # 80% dense
# #             0.15: (3, 4),   # 75% dense
# #             0.40: (2, 5),   # 40% dense
# #             0.50: (2, 4),   # 50% dense
# #         }
        
# #         # Find closest pattern
# #         closest_ratio = min(patterns.keys(), key=lambda x: abs(x - pruning_ratio))
# #         N, M = patterns[closest_ratio]
# #         return N, M
    
# #     # Function to get N:M pattern for a specific layer
# #     def get_nm_pattern(layer_name):
# #         """Determine N:M pattern based on layer name and group pruning ratios"""
# #         if not groups:
# #             return None, None  # Will use sparsity_ratio for uniform pruning
# #         if 'fc1' in layer_name or 'fc2' in layer_name or 'mlp' in layer_name:
# #             # MLP layers
# #             return groups.get('mlp_blocks', {}).pruning_ratio
# #         elif 'qkv' in layer_name:
# #             # QKV layers (part of attention)
# #             return groups.get('attention_blocks', {}).pruning_ratio
# #         elif ('proj' in layer_name and 'attn' in layer_name) or 'attention.output' in layer_name:
# #             # Attention projection layers
# #             return groups.get('output_projections', {}).pruning_ratio
# #         elif 'attention' in layer_name or 'attn' in layer_name:
# #             # General attention layers
# #             return groups.get('attention_blocks', {}).pruning_ratio
# #         else:
# #             return sparsity_ratio  # Default: use sparsity_ratio
    
# #     # Initialize input caches
# #     from maskllm.sparsegpt import SparseGPT
# #     layers = model.blocks
# #     dtype = next(iter(model.parameters())).dtype
# #     inps = torch.zeros(
# #         (nsamples, model.num_prefix_tokens+model.patch_embed.num_patches, model.embed_dim), dtype=dtype, device=device
# #     )
# #     cache = {'i': 0}

# #     class Catcher(nn.Module):
# #         def __init__(self, module):
# #             super().__init__()
# #             self.module = module
# #         def forward(self, inp, **kwargs):
# #             inps[cache['i']] = inp
# #             cache['i'] += 1
# #             raise ValueError
# #     layers[0] = Catcher(layers[0])
# #     for batch in loader:
# #         if cache['i'] == nsamples: break
# #         try:
# #             model(batch[0].to(device))
# #         except ValueError:
# #             pass
# #     layers[0] = layers[0].module
# #     torch.cuda.empty_cache()

# #     outs = torch.zeros_like(inps)
# #     print('Ready.')
    
# #     for i in range(len(layers)):
# #         layer = layers[i]
# #         print(layer)
# #         inps, outs = inps.to(device), outs.to(device)

# #         subset = find_layers(layer)

# #         gpts = {}
# #         for name in subset:
# #             gpts[name] = SparseGPT(subset[name])

# #         def add_batch(name):
# #             def tmp(_, inp, out):
# #                 gpts[name].add_batch(inp[0].data, out.data)
# #             return tmp

# #         handles = []
# #         for name in gpts:
# #             handles.append(subset[name].register_forward_hook(add_batch(name)))

# #         for j in range(nsamples):
# #             outs[j] = layer(inps[j].unsqueeze(0))[0]
# #         for h in handles:
# #             h.remove()

# #         for name in gpts:
# #             print(f"Pruning layer {i}, module: {name}")
# #             print('Pruning ...')
            
# #             # Get N:M pattern for this layer
# #             prune_n, prune_m = None, None
# #             sparsity_ratio = get_nm_pattern(name)
            
# #             if prune_n is not None and prune_m is not None:
# #                 # Use N:M pruning
# #                 print(f"  Using N:M pattern {prune_n}:{prune_m} for layer {name}")
# #                 gpts[name].fasterprune(
# #                     sparsity_ratio=0.0,  # Not used for N:M
# #                     prunen=prune_n, 
# #                     prunem=prune_m, 
# #                     percdamp=0.01, 
# #                     blocksize=128, 
# #                     disable_update=disable_update
# #                 )
# #             else:
# #                 # Use uniform sparsity ratio
# #                 print(f"  Using uniform sparsity {sparsity_ratio} for layer {name}")
# #                 gpts[name].fasterprune(
# #                     sparsity_ratio=sparsity_ratio,
# #                     prunen=0,  # Not used for uniform
# #                     prunem=0,  # Not used for uniform
# #                     percdamp=0.01, 
# #                     blocksize=128, 
# #                     disable_update=disable_update
# #                 )
            
# #             gpts[name].free()

# #         for j in range(nsamples):
# #             outs[j] = layer(inps[j].unsqueeze(0))[0]

# #         layers[i] = layer 
# #         torch.cuda.empty_cache()

# #         inps, outs = outs, inps

# #     torch.cuda.empty_cache()

# def prune_sparsegpt(model, loader, nsamples=128, batch_size=1, device=torch.device("cuda:0"), groups=None, sparsity_ratio=0.0, disable_update=False):
#     """
#     Prune Vision Transformer using SparseGPT with group-aware N:M patterns.
    
#     Args:
#         model: Vision Transformer model
#         loader: Data loader for calibration
#         nsamples: Number of calibration samples
#         groups: Isomorphic groups dictionary
#         sparsity_ratio: Default sparsity if no groups specified
#         disable_update: If True, only compute masks without weight update
#     """
#     model.to(device)
    
#     def ratio_to_nm(pruning_ratio):
#         """Convert pruning ratio to N:M pattern"""
#         # Calculate density (1 - pruning_ratio)
#         density = 1 - pruning_ratio
        
#         # Map density to appropriate N:M pattern
#         if density >= 0.83:   # ≤17% pruning
#             return (5, 6)   # 83.3% dense
#         elif density >= 0.8:  # 20% pruning
#             return (4, 5)   # 80% dense  
#         elif density >= 0.75: # 25% pruning
#             return (3, 4)   # 75% dense
#         elif density >= 0.5:  # 50% pruning
#             return (2, 4)   # 50% dense
#         elif density >= 0.4:  # 60% pruning
#             return (2, 5)   # 40% dense
#         elif density >= 0.25: # 75% pruning
#             return (1, 4)   # 25% dense
#         else:                 # No pruning or very high
#             return (0, 0)   # No N:M pruning, use uniform sparsity
    
#     def get_layer_config(layer_name):
#         """Get N:M pattern or sparsity ratio for a specific layer"""
#         if groups is None:
#             return None  # Use uniform sparsity
        
#         # Extract pruning ratio based on layer type
#         pruning_ratio = sparsity_ratio  # Default
        
#         # Check which group this layer belongs to
#         # Assuming groups is a dictionary of IsomorphicGroup objects
#         if 'mlp' in layer_name and ('fc1' in layer_name or 'fc2' in layer_name):
#             # MLP layers
#             if hasattr(groups, 'mlp_blocks'):
#                 pruning_ratio = groups.mlp_blocks.pruning_ratio
#             elif 'mlp_blocks' in groups:
#                 pruning_ratio = groups['mlp_blocks'].pruning_ratio
#         elif 'attn' in layer_name and ('qkv' in layer_name or 'proj' in layer_name):
#             # Attention layers
#             if hasattr(groups, 'attention_blocks'):
#                 pruning_ratio = groups.attention_blocks.pruning_ratio
#             elif 'attention_blocks' in groups:
#                 pruning_ratio = groups['attention_blocks'].pruning_ratio
#         elif 'head' in layer_name or ('fc' in layer_name and 'blocks' not in layer_name):
#             # Output head (classifier)
#             if hasattr(groups, 'output_projections'):
#                 pruning_ratio = groups.output_projections.pruning_ratio
#             elif 'output_projections' in groups:
#                 pruning_ratio = groups['output_projections'].pruning_ratio
        
#         return pruning_ratio
    
#     # Get all linear layers in the model
#     layers_dict = find_layers(model, layers=[nn.Linear])
#     layers = list(layers_dict.items())  # (name, module) pairs
    
#     # Initialize input caches
#     from maskllm.sparsegpt import SparseGPT
#     dtype = next(iter(model.parameters())).dtype
    
#     # For Vision Transformer, we need to get input dimension
#     # Assuming input shape: [batch, num_patches+1, embed_dim]
#     embed_dim = model.embed_dim
#     num_patches = model.patch_embed.num_patches
#     num_prefix_tokens = model.num_prefix_tokens  # cls token
    
#     inps = torch.zeros(
#         (nsamples, num_prefix_tokens + num_patches, embed_dim), 
#         dtype=dtype, 
#         device=device
#     )
    
#     cache = {'i': 0}

#     # Hook to capture inputs
#     class Catcher(nn.Module):
#         def __init__(self, module):
#             super().__init__()
#             self.module = module
#         def forward(self, inp, **kwargs):
#             inps[cache['i']] = inp
#             cache['i'] += 1
#             raise ValueError
    
#     # Replace first block with catcher to capture inputs
#     original_first_block = model.blocks[0]
#     model.blocks[0] = Catcher(original_first_block)
    
#     # Collect calibration data
#     for batch in loader:
#         if cache['i'] >= nsamples: 
#             break
#         try:
#             model(batch[0].to(device))
#         except ValueError:
#             pass
    
#     # Restore original first block
#     model.blocks[0] = original_first_block
#     torch.cuda.empty_cache()
    
#     print(f'Collected {cache["i"]} calibration samples')
    
#     # We need to run through the model to collect Hessians for each layer
#     # This is complex - let's use a simpler approach
    
#     # Instead, let's process each layer by running inputs through the model
#     # Store intermediate activations for each layer
    
#     # Create hooks to collect inputs for all layers
#     layer_inputs = {}
#     handles = []
    
#     def make_hook(name):
#         def hook(module, inp, out):
#             if name not in layer_inputs:
#                 layer_inputs[name] = []
#             layer_inputs[name].append(inp[0].detach())
#         return hook
    
#     # Register hooks for all linear layers
#     for name, layer in layers:
#         handles.append(layer.register_forward_hook(make_hook(name)))
    
#     # Run calibration data through model to collect inputs for each layer
#     print("Collecting layer inputs for Hessian calculation...")
#     with torch.no_grad():
#         for i, batch in enumerate(loader):
#             if i >= nsamples:
#                 break
#             model(batch[0].to(device))
    
#     # Remove hooks
#     for handle in handles:
#         handle.remove()
    
#     # Process each linear layer
#     for layer_name, layer in layers:
#         print(f'\nPruning layer: {layer_name}')
        
#         # Get configuration for this layer
#         pruning_ratio = get_layer_config(layer_name)
        
#         # Initialize SparseGPT for this layer
#         gpt = SparseGPT(layer)
        
#         # Add collected batches to SparseGPT
#         if layer_name in layer_inputs and len(layer_inputs[layer_name]) > 0:
#             # We need outputs too - this is tricky
#             # For simplicity, we'll run a forward pass for this specific layer
#             # A better approach would collect both inputs and outputs
            
#             # Temporary: use random data for calibration
#             # This is not ideal but works for demonstration
#             print(f"  Using {len(layer_inputs[layer_name])} batches for calibration")
            
#             # Create dummy outputs (not accurate but works for SparseGPT's Hessian approximation)
#             for inp in layer_inputs[layer_name][:min(10, len(layer_inputs[layer_name]))]:
#                 with torch.no_grad():
#                     out = layer(inp)
#                     gpt.add_batch(inp, out)
#         else:
#             print(f"  No inputs collected for {layer_name}, using random calibration")
#             # Fallback: use random data
#             for _ in range(min(10, nsamples)):
#                 dummy_inp = torch.randn(1, layer.in_features, device=device)
#                 with torch.no_grad():
#                     out = layer(dummy_inp)
#                     gpt.add_batch(dummy_inp, out)
        
#         # Determine pruning parameters
#         if pruning_ratio is None:
#             # Uniform sparsity
#             prune_n, prune_m = 0, 0
#             use_sparsity = sparsity_ratio
#             print(f'  Using uniform sparsity: {use_sparsity}')
#         else:
#             # N:M pruning based on group
#             prune_n, prune_m = ratio_to_nm(pruning_ratio)
            
#             if prune_n == 0 or prune_m == 0:
#                 # No N:M pruning, use uniform
#                 use_sparsity = pruning_ratio
#                 prune_n, prune_m = 0, 0
#                 print(f'  Using uniform sparsity: {use_sparsity} (no N:M pattern)')
#             else:
#                 use_sparsity = 0.0
#                 print(f'  Using N:M pattern {prune_n}:{prune_m} (from ratio {pruning_ratio})')
        
#         # Apply pruning
#         try:
#             gpt.fasterprune(
#                 sparsity_ratio=use_sparsity,
#                 prunen=prune_n,
#                 prunem=prune_m,
#                 percdamp=0.01,
#                 blocksize=128,
#                 disable_update=disable_update
#             )
#             print(f"  Successfully pruned {layer_name}")
#         except Exception as e:
#             print(f"  Error pruning {layer_name}: {e}")
#             # Fallback: skip this layer or apply simple magnitude pruning
#             if not disable_update:
#                 # Apply simple magnitude pruning
#                 weight = layer.weight.data
#                 if use_sparsity > 0:
#                     threshold = torch.quantile(weight.abs().flatten(), use_sparsity)
#                     mask = (weight.abs() > threshold).float()
#                     layer.weight.data *= mask
#                     print(f"  Applied magnitude pruning with sparsity {use_sparsity}")
        
#         gpt.free()
    
#     torch.cuda.empty_cache()
#     print('Pruning completed!')



In [19]:
prune_sparsegpt(model, val_loader, nsamples=128, batch_size=1, device=torch.device("cuda:0"), groups=groups)

Ready.
0 attn.qkv
Pruning ...
time 0.15
error 2544.22119140625
0 attn.proj
Pruning ...
time 0.04
error 176.33877563476562
0 mlp.fc1
Pruning ...
time 0.04
error 1119.798583984375
0 mlp.fc2
Pruning ...
time 0.17
error 47.41740036010742
1 attn.qkv
Pruning ...
time 0.04
error 8085.587890625
1 attn.proj
Pruning ...
time 0.04
error 213.3145294189453
1 mlp.fc1
Pruning ...
time 0.04
error 2103.24462890625
1 mlp.fc2
Pruning ...
time 0.17
error 52.827491760253906
2 attn.qkv
Pruning ...
time 0.04
error 11093.1328125
2 attn.proj
Pruning ...
time 0.04
error 329.5083923339844
2 mlp.fc1
Pruning ...
time 0.04
error 3215.494384765625
2 mlp.fc2
Pruning ...
time 0.17
error 80.84259033203125
3 attn.qkv
Pruning ...
time 0.04
error 13070.287109375
3 attn.proj
Pruning ...
time 0.04
error 311.3830871582031
3 mlp.fc1
Pruning ...
time 0.04
error 4007.045654296875
3 mlp.fc2
Pruning ...
time 0.19
error 84.6873779296875
4 attn.qkv
Pruning ...
time 0.04
error 16691.59375
4 attn.proj
Pruning ...
time 0.04
error 327.

In [20]:
count_parameters(model)

5526346

In [21]:
print_model_mac(model)

[📊] Total model MACs: 0.562G


In [23]:
import os
model_name = 'deit_tiny_patch16_224.fb_in1k'

for name, m in model.named_modules():
    if hasattr(m, 'mask'):
        print(f"Layer {name} Sparsity: {1 - torch.sum(m.mask).item()/m.mask.numel()}")
    
os.makedirs(os.path.dirname(f"output/pruned/{model_name}_sparsegpt_cifar10.pt"), exist_ok=True)
print(model)
torch.save(model.state_dict(), f"output/pruned/{model_name}_sparsegpt_cifar10.pt")
print(f"Model saved to output/pruned/{model_name}_sparsegpt_cifar10.pt")

Layer blocks.0.attn.qkv Sparsity: 0.7708333333333334
Layer blocks.0.attn.proj Sparsity: 0.7708333333333334
Layer blocks.0.mlp.fc1 Sparsity: 0.5
Layer blocks.0.mlp.fc2 Sparsity: 0.5
Layer blocks.1.attn.qkv Sparsity: 0.7708333333333334
Layer blocks.1.attn.proj Sparsity: 0.7708333333333334
Layer blocks.1.mlp.fc1 Sparsity: 0.5
Layer blocks.1.mlp.fc2 Sparsity: 0.5
Layer blocks.2.attn.qkv Sparsity: 0.7708333333333334
Layer blocks.2.attn.proj Sparsity: 0.7708333333333334
Layer blocks.2.mlp.fc1 Sparsity: 0.5
Layer blocks.2.mlp.fc2 Sparsity: 0.5
Layer blocks.3.attn.qkv Sparsity: 0.7708333333333334
Layer blocks.3.attn.proj Sparsity: 0.7708333333333334
Layer blocks.3.mlp.fc1 Sparsity: 0.5
Layer blocks.3.mlp.fc2 Sparsity: 0.5
Layer blocks.4.attn.qkv Sparsity: 0.7708333333333334
Layer blocks.4.attn.proj Sparsity: 0.7708333333333334
Layer blocks.4.mlp.fc1 Sparsity: 0.5
Layer blocks.4.mlp.fc2 Sparsity: 0.5
Layer blocks.5.attn.qkv Sparsity: 0.7708333333333334
Layer blocks.5.attn.proj Sparsity: 0.77083

# MaskLLM Training

In [24]:
maskllm_model = copy.deepcopy(original_model)

In [25]:
maskllm_model = replace_linear_with_(maskllm_model, MaskedLinear, exclude=[model.head], groups=groups)

Replaced blocks.0.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.0.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.0.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.0.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.1.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.1.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.1.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.1.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.2.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.2.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.2.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.2.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.3.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.3.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.3.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.3.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.4.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.4.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.4.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.4.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.5.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.5.attn.proj: {'N': 4, 'M': 5}
Replaced block

In [ ]:
from timm.optim import create_optimizer_v2, optimizer_kwargs